In [ ]:
# =============================================================================
# CLASSIFICATION TRAINING NOTEBOOK
# Predicting Molecular Stability for Organic Photovoltaics
# Best Training Pipeline - Optimal Parameters Hardcoded
# =============================================================================

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import Descriptors

from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

print("✓ All imports successful")

In [ ]:
# =============================================================================
# OPTIMAL PARAMETERS - Found via experimentation
# =============================================================================

# Best configuration: HOPV-only, LogisticRegression, k=40 features
# Kaggle F1 Score: 0.90

CONFIG = {
    'dataset': 'hopv_only',      # HOPV only (SQL excluded due to distribution mismatch)
    'k_features': 40,            # SelectKBest with f_classif
    'test_size': 0.2,
    'random_state': 42,
    'max_iter': 1000
}

print("Classification Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Load HOPV dataset only (SQL excluded due to class distribution mismatch)
df_hopv = pd.read_csv('../../../data/002_processed_data/xy_df_hopv.csv')
df_hopv = df_hopv.dropna(subset=['stability'])

print(f"HOPV dataset: {len(df_hopv)} samples")
print(f"\nClass distribution:")
print(df_hopv['stability'].value_counts())
print(f"\nClass balance: {df_hopv['stability'].value_counts().to_dict()}")

In [ ]:
# Load Kaggle test set
df_kaggle = pd.read_csv('../../../data/003_kaggle_competition_data/25261_regression_sample_submission.csv')

# Compute descriptors for Kaggle molecules
descriptor_names = [x[0] for x in Descriptors._descList]

def compute_descriptors(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return [np.nan] * len(descriptor_names)
    return [getattr(Descriptors, name)(mol) for name in descriptor_names]

X_kaggle = pd.DataFrame(
    [compute_descriptors(s) for s in df_kaggle['smiles'].tolist()],
    columns=descriptor_names
)

print(f"Kaggle test set: {len(df_kaggle)} samples")
print(f"Kaggle features: {X_kaggle.shape}")

In [ ]:
# Get descriptor columns
descriptor_cols = [col for col in df_hopv.columns if col not in ['smiles', 'homo', 'lumo', 'stability']]

X = df_hopv[descriptor_cols].values
y = df_hopv['stability'].values

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Features: {X.shape}")
print(f"Encoded classes: {dict(zip(le.classes_, range(len(le.classes_))))}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=CONFIG['test_size'],
    random_state=CONFIG['random_state'],
    stratify=y_encoded
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"\nTrain class distribution:")
for cls, count in zip(le.classes_, np.bincount(y_train)):
    print(f"  {cls}: {count}")
print(f"\nTest class distribution:")
for cls, count in zip(le.classes_, np.bincount(y_test)):
    print(f"  {cls}: {count}")

In [ ]:
print("="*60)
print("PREPROCESSING")
print("="*60)

# Variance threshold (remove constant features)
var_sel = VarianceThreshold(threshold=0)
X_train_clean = var_sel.fit_transform(X_train)
X_test_clean = var_sel.transform(X_test)
X_kaggle_clean = var_sel.transform(X_kaggle)

print(f"After variance threshold: {X_train_clean.shape[1]} features")

# Imputation
imp = SimpleImputer(strategy='median')
X_train_imp = imp.fit_transform(X_train_clean)
X_test_imp = imp.transform(X_test_clean)
X_kaggle_imp = imp.transform(X_kaggle_clean)

print(f"After imputation: {X_train_imp.shape}")

In [ ]:
print("="*60)
print("FEATURE SELECTION")
print("="*60)

# SelectKBest with f_classif (ANOVA F-value)
sel = SelectKBest(f_classif, k=CONFIG['k_features'])
X_train_sel = sel.fit_transform(X_train_imp, y_train)
X_test_sel = sel.transform(X_test_imp)
X_kaggle_sel = sel.transform(X_kaggle_imp)

print(f"Selected {CONFIG['k_features']} features")
print(f"Train shape: {X_train_sel.shape}")
print(f"Test shape: {X_test_sel.shape}")
print(f"Kaggle shape: {X_kaggle_sel.shape}")

In [ ]:
print("="*60)
print("SCALING")
print("="*60)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled = scaler.transform(X_test_sel)
X_kaggle_scaled = scaler.transform(X_kaggle_sel)

print(f"✓ Data scaled")
print(f"Train: {X_train_scaled.shape}")
print(f"Test: {X_test_scaled.shape}")
print(f"Kaggle: {X_kaggle_scaled.shape}")

In [ ]:
print("="*60)
print("TRAINING LOGISTIC REGRESSION")
print("="*60)

# Initialize model
model = LogisticRegression(
    max_iter=CONFIG['max_iter'],
    random_state=CONFIG['random_state']
)

# Cross-validation
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='f1_macro')

print(f"CV F1 scores per fold:")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"\nCV F1 mean: {cv_scores.mean():.4f}")
print(f"CV F1 std:  {cv_scores.std():.4f}")

# Train on full training set
model.fit(X_train_scaled, y_train)
print("\n✓ Model trained")

In [ ]:
print("="*60)
print("TEST SET EVALUATION")
print("="*60)

# Predict on test set
test_pred = model.predict(X_test_scaled)
test_f1 = f1_score(y_test, test_pred, average='macro')

print(f"Test F1 Score: {test_f1:.4f}")

print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, test_pred)
print(cm)

print(f"\nClassification Report:")
print(classification_report(y_test, test_pred, target_names=le.classes_))

In [ ]:
print("="*60)
print("SAVING MODEL WITH JOBLIB")
print("="*60)

os.makedirs('../../../models', exist_ok=True)

model_package = {
    # Preprocessing
    'var_sel': var_sel,
    'imp': imp,
    'sel': sel,
    'scaler': scaler,
    'le': le,
    'descriptor_cols': descriptor_cols,
    
    # Model
    'model': model,
    'config': CONFIG,
    
    # Performance metrics
    'cv_f1_mean': cv_scores.mean(),
    'cv_f1_std': cv_scores.std(),
    'test_f1': test_f1,
    
}

joblib.dump(model_package, '../../../models/classification_model.joblib')

print("✓ Model saved to: ../../../models/classification_model.joblib")